# 🚁 UAV Detection Training - YOLOv8 + ViT
**Notebook này train mô hình phát hiện UAV/drone sử dụng YOLOv8 tích hợp Vision Transformer (ViT)**

- **Dataset**: Drone Detection Dataset (Roboflow Universe - công khai, ~1000 ảnh)
- **Model**: YOLOv8n với custom ViT backbone
- **Thời gian dự kiến**: ~30 phút trên GPU T4 (50 epochs)

> ⚠️ **Trước khi chạy**: Vào `Runtime > Change runtime type > T4 GPU`

## 📦 Bước 1: Cài đặt thư viện

In [ ]:
# Cài đặt Ultralytics và Roboflow
!pip install ultralytics roboflow --quiet

import ultralytics
print(f'✅ Ultralytics version: {ultralytics.__version__}')

import torch
print(f'✅ PyTorch version: {torch.__version__}')
print(f'✅ GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

## 📂 Bước 2: Tải Dataset (Drone Detection - Roboflow)

Dataset công khai: **"Drone Detection" từ Roboflow Universe**
- ~1000 ảnh drone chụp từ nhiều góc độ
- 1 class: `drone`
- Format: YOLOv8

In [ ]:
import os

# === OPTION A: Dùng Roboflow dataset (cần API key - FREE) ===
# Đăng ký tại https://roboflow.com để lấy API key miễn phí

USE_ROBOFLOW = True  # Đặt False nếu muốn tự upload dataset

if USE_ROBOFLOW:
    from roboflow import Roboflow
    
    # API key miễn phí - đăng ký tại roboflow.com
    # HOẶC dùng key này (demo key, có thể hết hạn):
    rf = Roboflow(api_key="YOUR_API_KEY_HERE")
    
    # Dataset: Drone Detection Dataset
    project = rf.workspace("drone-vuoxt").project("drone-detection-frhon")
    version = project.version(1)
    dataset = version.download("yolov8")
    
    DATA_YAML = dataset.location + "/data.yaml"
    print(f"✅ Dataset downloaded to: {dataset.location}")
    print(f"📄 YAML: {DATA_YAML}")
else:
    # === OPTION B: Tự generate dataset synthetic (không cần internet) ===
    print("⚠️ Sử dụng OPTION B - xem cell tiếp theo")

In [ ]:
# === OPTION B: Tạo dataset synthetic đơn giản (không cần Roboflow) ===
# Chỉ chạy cell này nếu OPTION A gặp lỗi

import os
import numpy as np
from PIL import Image, ImageDraw
import random
import yaml

def create_synthetic_drone_dataset(base_dir="/content/drone_dataset", 
                                    n_train=200, n_val=50):
    """Tạo dataset synthetic: ảnh nền ngẫu nhiên + bounding box drone giả"""
    
    splits = {"train": n_train, "val": n_val}
    
    for split, n in splits.items():
        img_dir = f"{base_dir}/images/{split}"
        lbl_dir = f"{base_dir}/labels/{split}"
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(lbl_dir, exist_ok=True)
        
        for i in range(n):
            # Tạo ảnh nền (sky-like color)
            w, h = 640, 640
            sky_color = (
                random.randint(100, 200),
                random.randint(150, 220),
                random.randint(200, 255)
            )
            img = Image.new("RGB", (w, h), sky_color)
            draw = ImageDraw.Draw(img)
            
            # Thêm noise nền
            for _ in range(random.randint(5, 20)):
                x0 = random.randint(0, w-50)
                y0 = random.randint(0, h-50)
                cloud_color = tuple(c + random.randint(-30, 30) for c in sky_color)
                draw.ellipse([x0, y0, x0+random.randint(30, 100), y0+random.randint(20, 60)],
                             fill=cloud_color)
            
            # Thêm 1-3 drone objects
            n_drones = random.randint(1, 3)
            labels = []
            
            for _ in range(n_drones):
                # Kích thước drone (nhỏ, đặc trưng UAV)
                dw = random.randint(20, 80)
                dh = random.randint(15, 60)
                cx = random.randint(dw//2, w - dw//2)
                cy = random.randint(dh//2, h - dh//2)
                
                # Vẽ drone (hình chữ thập đơn giản)
                drone_color = tuple(random.randint(20, 80) for _ in range(3))
                # Body
                draw.rectangle([cx-dw//4, cy-dh//4, cx+dw//4, cy+dh//4], 
                                fill=drone_color)
                # Arms
                draw.rectangle([cx-dw//2, cy-3, cx+dw//2, cy+3], fill=drone_color)
                draw.rectangle([cx-3, cy-dh//2, cx+3, cy+dh//2], fill=drone_color)
                # Rotors
                for rx, ry in [(cx-dw//2, cy-dh//2), (cx+dw//2, cy-dh//2),
                                (cx-dw//2, cy+dh//2), (cx+dw//2, cy+dh//2)]:
                    draw.ellipse([rx-8, ry-4, rx+8, ry+4], fill=(150, 150, 150))
                
                # YOLO format: class cx cy w h (normalized)
                labels.append(f"0 {cx/w:.6f} {cy/h:.6f} {dw/w:.6f} {dh/h:.6f}")
            
            # Lưu ảnh và label
            img.save(f"{img_dir}/drone_{i:04d}.jpg")
            with open(f"{lbl_dir}/drone_{i:04d}.txt", "w") as f:
                f.write("\n".join(labels))
        
        print(f"✅ Created {n} {split} images")
    
    # Tạo dataset.yaml
    yaml_content = {
        "path": base_dir,
        "train": "images/train",
        "val": "images/val",
        "nc": 1,
        "names": ["uav"]
    }
    yaml_path = f"{base_dir}/dataset.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(yaml_content, f, default_flow_style=False)
    
    print(f"\n✅ Dataset created at: {base_dir}")
    print(f"📄 YAML: {yaml_path}")
    return yaml_path


# Chạy tạo dataset synthetic
DATA_YAML = create_synthetic_drone_dataset(
    base_dir="/content/drone_dataset",
    n_train=300,   # Số ảnh train
    n_val=80       # Số ảnh validation
)
print(f"\n🎯 DATA_YAML = {DATA_YAML}")

## 🔧 Bước 3: Định nghĩa Custom ViT Module

Module Vision Transformer (ViT) tích hợp vào YOLO backbone để tăng khả năng nhận diện UAV nhỏ trong ảnh drone.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ViT(nn.Module):
    """
    Lightweight Vision Transformer cho YOLO integration.
    Dùng 1x1 Conv thay vì patch embedding để giữ resolution.
    
    Args:
        c1: Input channels (tự động nhận từ YOLO)
        c2: Output channels
        num_heads: Số attention heads (default: 4)
        depth: Số transformer blocks (default: 2)
    """
    
    def __init__(self, c1, c2, num_heads=4, depth=2):
        super().__init__()
        self.c1 = c1
        self.c2 = c2
        embed_dim = c2
        
        # Input projection (1x1 conv để giữ resolution)
        self.input_proj = nn.Conv2d(c1, embed_dim, kernel_size=1)
        self.norm_pre = nn.LayerNorm(embed_dim)
        
        # Transformer blocks
        self.blocks = nn.ModuleList()
        for _ in range(depth):
            self.blocks.append(nn.ModuleDict({
                'norm1': nn.LayerNorm(embed_dim),
                'attn': nn.MultiheadAttention(embed_dim, num_heads, batch_first=True),
                'norm2': nn.LayerNorm(embed_dim),
                'mlp': nn.Sequential(
                    nn.Linear(embed_dim, embed_dim * 4),
                    nn.GELU(),
                    nn.Linear(embed_dim * 4, embed_dim)
                )
            }))
        
        # Output projection
        self.output_proj = nn.Conv2d(embed_dim, c2, kernel_size=1)
    
    def forward(self, x):
        B, C, H, W = x.shape
        
        # Project input
        x = self.input_proj(x)       # (B, embed_dim, H, W)
        
        # Flatten to sequence
        x = x.flatten(2).transpose(1, 2)  # (B, H*W, embed_dim)
        x = self.norm_pre(x)
        
        # Transformer blocks
        for block in self.blocks:
            x_norm = block['norm1'](x)
            attn_out, _ = block['attn'](x_norm, x_norm, x_norm)
            x = x + attn_out
            x = x + block['mlp'](block['norm2'](x))
        
        # Reshape back to feature map
        x = x.transpose(1, 2).reshape(B, self.c2, H, W)
        x = self.output_proj(x)
        
        return x


def register_custom_modules():
    """Đăng ký custom modules vào Ultralytics."""
    from ultralytics.nn import modules
    import ultralytics.nn.modules.block as block_module
    import ultralytics.nn.tasks as tasks
    
    setattr(modules, 'ViT', ViT)
    setattr(block_module, 'ViT', ViT)
    setattr(tasks, 'ViT', ViT)
    
    print("✅ Custom modules registered: ViT")


register_custom_modules()

## 📐 Bước 4: Tạo Model Config (YAML)

Tạo file config cho **YOLOv8n + ViT** - thêm ViT branch vào backbone để học global attention.

In [ ]:
# === CONFIG 1: YOLOv8n thuần (baseline - train nhanh hơn) ===
MODEL_BASELINE = "yolov8n.pt"  # Pretrained YOLOv8n


# === CONFIG 2: YOLOv8n + ViT Branch (custom architecture) ===
vit_yaml_content = """# YOLOv8n + ViT Branch - UAV Detection
nc: 1  # số classes
names: ['uav']

# ------------- BACKBONE YOLOv8n -------------
backbone:
  - [-1, 1, Conv, [16, 3, 2]]           # 0: 320x320, 16ch
  - [-1, 1, Conv, [32, 3, 2]]           # 1: 160x160, 32ch
  - [-1, 1, C2f, [32, True]]            # 2
  - [-1, 1, Conv, [64, 3, 2]]           # 3: 80x80
  - [-1, 2, C2f, [64, True]]            # 4
  - [-1, 1, Conv, [128, 3, 2]]          # 5: 40x40
  - [-1, 2, C2f, [128, True]]           # 6
  - [-1, 1, Conv, [256, 3, 2]]          # 7: 20x20
  - [-1, 1, C2f, [256, True]]           # 8
  - [-1, 1, SPPF, [256, 5]]             # 9

# ------------- ViT BRANCH (trên feature P3) -------------
  - [4, 1, ViT, [64, 4, 1]]            # 10: ViT trên P3 (80x80), out 64ch

# ------------- HEAD -------------
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]  # 11: upsample ViT feat
  - [9, 1, nn.Upsample, [None, 2, 'nearest']]   # 12: upsample P5
  - [[-1, 6], 1, Concat, [1]]                    # 13: cat P5+P4
  - [-1, 1, C2f, [128, False]]                    # 14

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]   # 15
  - [[-1, 4, 11], 1, Concat, [1]]                # 16: cat P4+P3+ViT
  - [-1, 1, C2f, [64, False]]                     # 17 (P3/8 output)

  - [-1, 1, Conv, [64, 3, 2]]                     # 18
  - [[-1, 14], 1, Concat, [1]]                    # 19
  - [-1, 1, C2f, [128, False]]                    # 20 (P4/16 output)

  - [-1, 1, Conv, [128, 3, 2]]                    # 21
  - [[-1, 9], 1, Concat, [1]]                     # 22
  - [-1, 1, C2f, [256, False]]                    # 23 (P5/32 output)

  - [[17, 20, 23], 1, Detect, [nc]]               # 24: Detect P3, P4, P5
"""

with open("/content/yolov8n_vit.yaml", "w") as f:
    f.write(vit_yaml_content)

print("✅ Model config saved: /content/yolov8n_vit.yaml")
print("\n📐 Architecture: YOLOv8n Backbone + ViT Branch trên P3 (80x80)")
print("   - ViT giúp học global attention cho object nhỏ như drone")
print("   - Concat ViT features vào P3 head")

## 🔍 Bước 5: Kiểm tra Dataset

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np
import glob

# Đếm số ảnh
train_imgs = glob.glob("/content/drone_dataset/images/train/*.jpg")
val_imgs = glob.glob("/content/drone_dataset/images/val/*.jpg")
print(f"📊 Train images: {len(train_imgs)}")
print(f"📊 Val images: {len(val_imgs)}")

# Hiển thị mẫu
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
sample_imgs = train_imgs[:8]

for ax, img_path in zip(axes.flat, sample_imgs):
    img = Image.open(img_path)
    label_path = img_path.replace("images", "labels").replace(".jpg", ".txt")
    
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path)[:15], fontsize=8)
    ax.axis("off")
    
    # Vẽ bounding boxes
    if os.path.exists(label_path):
        w, h = img.size
        with open(label_path) as f:
            for line in f:
                cls, cx, cy, bw, bh = map(float, line.strip().split())
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                          linewidth=2, edgecolor='red', facecolor='none')
                ax.add_patch(rect)

plt.suptitle("🚁 Dataset Samples (Red = UAV bounding box)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("/content/dataset_samples.png", dpi=100, bbox_inches='tight')
plt.show()
print("✅ Dataset visualization saved")

## 🏋️ Bước 6: Training

Chạy 2 experiments:
1. **Baseline**: YOLOv8n pretrained (nhanh, reference)
2. **Custom**: YOLOv8n + ViT Branch (model của dự án NCKH)

In [ ]:
# ============================================================
# EXPERIMENT 1: YOLOv8n Baseline (Pretrained)
# ============================================================
from ultralytics import YOLO
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Device: {device}")

print("\n" + "="*60)
print("🚀 EXPERIMENT 1: YOLOv8n Baseline")
print("="*60)

model_baseline = YOLO("yolov8n.pt")

results_baseline = model_baseline.train(
    data=DATA_YAML,
    epochs=50,              # 50 epochs để demo nhanh
    imgsz=640,
    batch=16,
    device=device,
    project="/content/runs",
    name="yolov8n_baseline",
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    momentum=0.9,
    weight_decay=0.0005,
    patience=20,
    save_period=10,
    workers=2,
    amp=True,
    cos_lr=True,
    mosaic=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    verbose=True
)

print("\n✅ Baseline training hoàn tất!")

In [ ]:
# ============================================================
# EXPERIMENT 2: YOLOv8n + ViT Branch (Custom)
# ============================================================

# Đăng ký lại custom modules (phòng trường hợp kernel restart)
register_custom_modules()

print("\n" + "="*60)
print("🚀 EXPERIMENT 2: YOLOv8n + ViT Branch")
print("="*60)

model_vit = YOLO("/content/yolov8n_vit.yaml")

# Kiểm tra model
model_info = model_vit.info()
print(f"\n📊 Model parameters: {sum(p.numel() for p in model_vit.model.parameters()):,}")

results_vit = model_vit.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project="/content/runs",
    name="yolov8n_vit",
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    momentum=0.9,
    weight_decay=0.0005,
    patience=20,
    save_period=10,
    workers=2,
    amp=True,
    cos_lr=True,
    mosaic=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    verbose=True
)

print("\n✅ YOLOv8n + ViT training hoàn tất!")

## 📊 Bước 7: So sánh kết quả

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def load_results(run_name):
    csv_path = f"/content/runs/detect/{run_name}/results.csv"
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        return df
    return None

df_baseline = load_results("yolov8n_baseline")
df_vit = load_results("yolov8n_vit")

# ===== VẼ BIỂU ĐỒ SO SÁNH =====
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("📊 So sánh YOLOv8n Baseline vs YOLOv8n + ViT", 
             fontsize=16, fontweight='bold')

metrics = [
    ("metrics/mAP50(B)", "mAP@50"),
    ("metrics/mAP50-95(B)", "mAP@50-95"),
    ("metrics/precision(B)", "Precision"),
    ("metrics/recall(B)", "Recall"),
    ("train/box_loss", "Train Box Loss"),
    ("val/box_loss", "Val Box Loss"),
]

colors = {"Baseline": "#2196F3", "ViT": "#FF5722"}

for ax, (col, title) in zip(axes.flat, metrics):
    if df_baseline is not None and col in df_baseline.columns:
        ax.plot(df_baseline["epoch"], df_baseline[col], 
                color=colors["Baseline"], linewidth=2, label="YOLOv8n Baseline")
    if df_vit is not None and col in df_vit.columns:
        ax.plot(df_vit["epoch"], df_vit[col],
                color=colors["ViT"], linewidth=2, label="YOLOv8n + ViT")
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/training_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Comparison chart saved: /content/training_comparison.png")

In [ ]:
# ===== BẢNG KẾT QUẢ CUỐI =====
print("\n" + "="*65)
print("📋 KẾT QUẢ CUỐI CÙNG (epoch cuối)")
print("="*65)

summary_data = []

for name, df in [("YOLOv8n Baseline", df_baseline), ("YOLOv8n + ViT", df_vit)]:
    if df is not None:
        last = df.iloc[-1]
        row = {
            "Model": name,
            "mAP@50": f"{last.get('metrics/mAP50(B)', 0):.4f}",
            "mAP@50-95": f"{last.get('metrics/mAP50-95(B)', 0):.4f}",
            "Precision": f"{last.get('metrics/precision(B)', 0):.4f}",
            "Recall": f"{last.get('metrics/recall(B)', 0):.4f}",
        }
        summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Tính cải thiện
if len(summary_data) == 2:
    b_map = float(summary_data[0]['mAP@50'])
    v_map = float(summary_data[1]['mAP@50'])
    improvement = (v_map - b_map) / (b_map + 1e-8) * 100
    print(f"\n📈 Cải thiện mAP@50 của ViT so với Baseline: {improvement:+.2f}%")

print("\n" + "="*65)

## 🔮 Bước 8: Inference (Kiểm tra dự đoán)

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np
import glob

# Load best model
best_model_path = "/content/runs/detect/yolov8n_vit/weights/best.pt"
if not os.path.exists(best_model_path):
    best_model_path = "/content/runs/detect/yolov8n_baseline/weights/best.pt"

print(f"🔮 Loading model: {best_model_path}")
register_custom_modules()  # Đăng ký lại trước khi load
best_model = YOLO(best_model_path)

# Lấy ảnh val để test
val_imgs = sorted(glob.glob("/content/drone_dataset/images/val/*.jpg"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("🚁 UAV Detection - Inference Results", fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flat, val_imgs):
    # Predict
    results = best_model.predict(img_path, conf=0.25, verbose=False)
    result = results[0]
    
    # Vẽ kết quả
    img = Image.open(img_path)
    ax.imshow(img)
    
    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0])
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                      linewidth=2.5, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-5, f"UAV {conf:.2f}", 
                    color='lime', fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.5))
        ax.set_title(f"✅ {len(result.boxes)} detected", color='green')
    else:
        ax.set_title("❌ No detection", color='red')
    
    ax.axis("off")

plt.tight_layout()
plt.savefig("/content/inference_results.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Inference results saved: /content/inference_results.png")

## 💾 Bước 9: Lưu kết quả về Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Tạo thư mục lưu trên Drive
save_dir = "/content/drive/MyDrive/UAV_NCKH_Results"
os.makedirs(save_dir, exist_ok=True)

# Copy toàn bộ runs
if os.path.exists("/content/runs"):
    shutil.copytree("/content/runs", f"{save_dir}/runs", dirs_exist_ok=True)
    print("✅ Runs saved to Drive")

# Copy ảnh kết quả
for fname in ["dataset_samples.png", "training_comparison.png", "inference_results.png"]:
    src = f"/content/{fname}"
    if os.path.exists(src):
        shutil.copy(src, save_dir)
        print(f"✅ Saved: {fname}")

print(f"\n🎉 Tất cả kết quả đã lưu vào: {save_dir}")
print(f"   - runs/detect/yolov8n_baseline/weights/best.pt")
print(f"   - runs/detect/yolov8n_vit/weights/best.pt")
print(f"   - training_comparison.png")
print(f"   - inference_results.png")

## 📝 Ghi chú & Next Steps

### 🔄 Để dùng dataset thật của dự án:
1. Upload dataset lên Google Drive
2. Mount Drive và thay `DATA_YAML` bằng đường dẫn thực tế:
   ```python
   DATA_YAML = "/content/drive/MyDrive/UAV_Dataset/dataset.yaml"
   ```

### ⬆️ Để scale up:
- Tăng `epochs=200` cho training đầy đủ
- Thay `yolov8n` → `yolov8s` hoặc `yolov8m` để mô hình lớn hơn
- Tăng `batch=32` nếu GPU VRAM đủ lớn (A100)

### 📌 Dataset thật (Roboflow):
- [Drone Detection Dataset](https://universe.roboflow.com/drone-vuoxt/drone-detection-frhon)
- [UAV Detection (VisDrone)](https://universe.roboflow.com/visdrone/visdrone-dataset)